In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

documents = [
    "The Reserve Bank of India kept interest rates unchanged citing inflation concerns.",
    "Manchester United signed a new striker in the winter transfer window.",
    "Researchers published a new paper on transformer attention mechanisms.",
    "The central bank held its policy rate steady for the third straight meeting.",
    "A new striker joined the football club ahead of the upcoming season.",
    "Self-attention allows models to weigh the importance of different tokens.",
    "Heavy rainfall caused flooding in several districts of Assam this week.",
    "The football team's new signing scored on his debut match.",
    "BERT and GPT are both built on the transformer architecture.",
    "Monsoon floods displaced thousands of families in the northeastern states."
]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
doc_embeddings = embed_model.encode(documents, convert_to_tensor=False)
doc_embeddings = np.array(doc_embeddings).astype("float32")
print(doc_embeddings.shape)

(10, 384)


In [3]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# Normalize vectors so inner product = cosine similarity
faiss.normalize_L2(doc_embeddings)
index.add(doc_embeddings)

print("Number of vectors in index:", index.ntotal)

Number of vectors in index: 10


In [4]:
query = "RBI holds repo rate steady amid inflation worries"
query_embedding = embed_model.encode([query], convert_to_tensor=False)
query_embedding = np.array(query_embedding).astype("float32")
faiss.normalize_L2(query_embedding)

k = 3  # top 3 results
scores, indices = index.search(query_embedding, k)

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    print(f"{rank}. (score: {score:.4f}) {documents[idx]}")

1. (score: 0.5614) The Reserve Bank of India kept interest rates unchanged citing inflation concerns.
2. (score: 0.4547) The central bank held its policy rate steady for the third straight meeting.
3. (score: 0.1376) Heavy rainfall caused flooding in several districts of Assam this week.


In [5]:
query2 = "New forward joins soccer club before season starts"
query2_embedding = embed_model.encode([query2], convert_to_tensor=False)
query2_embedding = np.array(query2_embedding).astype("float32")
faiss.normalize_L2(query2_embedding)

scores2, indices2 = index.search(query2_embedding, k)

for rank, (idx, score) in enumerate(zip(indices2[0], scores2[0]), start=1):
    print(f"{rank}. (score: {score:.4f}) {documents[idx]}")

1. (score: 0.6922) A new striker joined the football club ahead of the upcoming season.
2. (score: 0.4525) Manchester United signed a new striker in the winter transfer window.
3. (score: 0.4015) The football team's new signing scored on his debut match.


In [7]:
query3 = "best recipe for making chocolate cake"
query3_embedding = embed_model.encode([query3], convert_to_tensor=False)
query3_embedding = np.array(query3_embedding).astype("float32")
faiss.normalize_L2(query3_embedding)

scores3, indices3 = index.search(query3_embedding, k)

for rank, (idx, score) in enumerate(zip(indices3[0], scores3[0]), start=1):
    print(f"{rank}. (score: {score:.4f}) {documents[idx]}")

1. (score: 0.0552) Self-attention allows models to weigh the importance of different tokens.
2. (score: 0.0440) The Reserve Bank of India kept interest rates unchanged citing inflation concerns.
3. (score: 0.0292) Heavy rainfall caused flooding in several districts of Assam this week.
